# guid

In [ ]:
open sm'_operators

In [ ]:
///- --test static

open testing

## guid

### guid

In [ ]:
nominal guid_python =
    `(
        global "import uuid"
        $'' : $'uuid.UUID'
    )
type guid_switch =
    {
        Gleam : string
        Lua : string
        Fsharp : $'System.Guid'
        Python : guid_python
        Rust : $'Rc<str>'
        TypeScript : string
        Zig : string
    }
nominal guid = $'backend_switch `(guid_switch)'

### new_guid

In [ ]:
inl new_guid (x : string) : guid =
    run_target_args (fun () => x) function
        | Zig => fun _ => $'(std.ascii.allocLowerString(spiral_gpa, !x) catch \@panic("out of memory"))' : guid
        | Gleam => fun x =>
            inl lower = sm'.to_lower x
            $'!lower '
        | Lua => fun x =>
            inl lower = sm'.to_lower x
            $'!lower '
        | Cuda => fun x =>
            global "import uuid"
            inl lower = sm'.to_lower x
            $'!lower '
        | Rust => fun x => $'Rc::<str>::from(!x.to_lowercase())'
        | TypeScript => fun x => $'!x.toLowerCase()'
        | _ => fun x => $'!x |> `guid '

### to_string

In [ ]:
inl to_string (x : guid) : string =
    run_target_args (fun () => x) function
        | Fsharp => fun x => $'!x.ToString()' : string
        | _ => fun x => $'!x ' : string

### zig_random_u64

In [ ]:
inl zig_random_u64 () : u64 =
    global """extern "kernel32" fn QueryPerformanceCounter(count: *i64) callconv(.winapi) c_int;"""
    global """
        var spiral_random_state: u64 = 0;
        fn spiralRandomU64() u64 {
            if (spiral_random_state == 0) {
                var seed: u64 = 0;
                if (builtin.os.tag == .windows) {
                    var count: i64 = 0;
                    _ = QueryPerformanceCounter(&count);
                    seed = @bitCast(count);
                } else if (builtin.os.tag == .linux) {
                    var ts: std.os.linux.timespec = undefined;
                    _ = std.os.linux.clock_gettime(.MONOTONIC, &ts);
                    seed = @as(u64, @bitCast(@as(i64, ts.sec))) *% 1000000007 +% @as(u64, @bitCast(@as(i64, ts.nsec)));
                }
                spiral_random_state = (seed ^ @intFromPtr(&spiral_random_state)) | 1;
            }
            var x = spiral_random_state;
            x ^= x << 13;
            x ^= x >> 7;
            x ^= x << 17;
            spiral_random_state = x;
            return x;
        }
        """
    $'spiralRandomU64()' : u64

### new_raw_guid

In [ ]:
inl new_raw_guid () : guid =
    backend_switch {
        Gleam = fun () =>
            global "import gleam_uuid"
            $'gleam_uuid.v4_string ()' : guid
        Lua = fun () => $'""' : guid
        Fsharp = fun () => $'System.Guid.NewGuid' () : guid
        TypeScript = fun () => $'globalThis.crypto.randomUUID()' : guid
        Python = fun () =>
            global "import uuid"
            $'uuid.uuid4()' : guid
        Zig = fun () =>
            inl a = zig_random_u64 ()
            inl b = zig_random_u64 ()
            $'(std.fmt.allocPrint(spiral_gpa, "{x:0>8}-{x:0>4}-4{x:0>3}-{x:0>4}-{x:0>12}", .{ !a & 0xffffffff, (!a >> 32) & 0xffff, (!a >> 48) & 0xfff, (!b & 0x3fff) | 0x8000, (!b >> 16) & 0xffffffffffff }) catch \@panic("out of memory"))' : guid
        Rust = fun () => $'{ use std::hash::{BuildHasher, Hasher}; let mut h = std::collections::hash_map::RandomState::new().build_hasher(); h.write_u128(std::time::SystemTime::now().duration_since(std::time::UNIX_EPOCH).map(|d| d.as_nanos()).unwrap_or(0)); let a = h.finish(); h.write_u64(a); let b = h.finish(); let a = (a & 0xffff_ffff_ffff_0fff) | 0x4000; let b = (b & 0x3fff_ffff_ffff_ffff) | 0x8000_0000_0000_0000; Rc::<str>::from(format\!("{:08x}-{:04x}-{:04x}-{:04x}-{:012x}", a >> 32, (a >> 16) & 0xffff, a & 0xffff, b >> 48, b & 0xffff_ffff_ffff)) }' : guid
    }

### hash_guid

In [ ]:
type hash_guid = guid

let hash_guid (hash : string) : hash_guid =
    inl body () =
        inl dash = "-"
        join
            inl hash = hash |> sm'.pad_left 32i32 '0'
            inl a = hash |> sm'.range (am'.Start 0i32) (am'.End fun _ => 8)
            inl b = hash |> sm'.range (am'.Start 8i32) (am'.End fun _ => 12)
            inl c = hash |> sm'.range (am'.Start 12i32) (am'.End fun _ => 16)
            inl d = hash |> sm'.range (am'.Start 16i32) (am'.End fun _ => 20)
            inl e = hash |> sm'.range (am'.Start 20i32) (am'.End fun _ => 32)
            a ++# dash ++# b ++# dash ++# c ++# dash ++# d ++# dash ++# e
            |> new_guid
    run_target function
        | Zig => fun () => body ()
        | _ => body

In [1]:
///- --test
///> gleam
///> fsharp
///> cuda
///> rust
///> typescript

""
|> hash_guid
|> _assert_eq' ("00000000-0000-0000-0000-000000000000" |> new_guid)

"123456789012345678901234567890123"
|> hash_guid
|> _assert_eq' ("12345678-9012-3456-7890-123456789012" |> new_guid)

.gleam (gleam)
__assert_eq' { expected = "00000000-0000-0000-0000-000000000000" }
__assert_eq' { expected = "12345678-9012-3456-7890-123456789012" }
.fsx (fsharp)
__assert_eq' { expected = 00000000-0000-0000-0000-000000000000 }
__assert_eq' { expected = 12345678-9012-3456-7890-123456789012 }
.py (cuda)
__assert_eq' { expected = 00000000-0000-0000-0000-000000000000 }
__assert_eq' { expected = 12345678-9012-3456-7890-123456789012 }
.rs (rust)
__assert_eq' { expected = "00000000-0000-0000-0000-000000000000" }
__assert_eq' { expected = "12345678-9012-3456-7890-123456789012" }
.ts (typescript)
__assert_eq' { expected = 00000000-0000-0000-0000-000000000000 }
__assert_eq' { expected = 12345678-9012-3456-7890-123456789012 }

0

#### to_string_is_lowercase_text (test)

In [2]:
///- --test
///> fsharp
///> rust
///> zig
///> cuda
///> gleam
///> lua

"6543210F-EDCB-A987-6543-210FEDCBA987"
|> new_guid
|> to_string
|> _assert_eq "6543210f-edcb-a987-6543-210fedcba987"

.fsx (fsharp)
__assert_eq { expected = 6543210f-edcb-a987-6543-210fedcba987 }
.rs (rust)
__assert_eq { expected = 6543210f-edcb-a987-6543-210fedcba987 }
.zig (zig)
__assert_eq { expected = 6543210f-edcb-a987-6543-210fedcba987 }
.py (cuda)
__assert_eq { expected = 6543210f-edcb-a987-6543-210fedcba987 }
.gleam (gleam)
__assert_eq { expected = 6543210f-edcb-a987-6543-210fedcba987 }
.lua (lua)
__assert_eq { expected = 6543210f-edcb-a987-6543-210fedcba987 }

0

## main

In [3]:
///> fsharp

inl main () =
    $'let new_guid x = !new_guid x' : ()
    $'let hash_guid x = !hash_guid x' : ()
    $'let new_raw_guid x = !new_raw_guid x' : ()